# WildDex Model Training
**EfficientNetB0 transfer learning — 50 species**

Steps:
1. Install deps
2. Download images from iNaturalist (~200/class)
3. Train EfficientNetB0 (2-phase)
4. Convert to TFLite (float16)
5. Download `wilddex_model.tflite` + `labels.txt`

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q requests tensorflow

In [ ]:
# ── 2. Define 50 species ─────────────────────────────────────────────────────
# Keys = folder names (become model labels, sorted alphabetically)
# Values = iNaturalist taxon name (scientific or common)

INAT_NAMES = {
    'bald_eagle':        'Haliaeetus leucocephalus',
    'canada_goose':      'Branta canadensis',
    'cat':               'Felis catus',
    'chameleon':         'Chamaeleonidae',
    'cheetah':           'Acinonyx jubatus',
    'chicken':           'Gallus gallus domesticus',
    'chimpanzee':        'Pan troglodytes',
    'cow':               'Bos taurus',
    'crocodile':         'Crocodylidae',
    'crow':              'Corvus brachyrhynchos',
    'dog':               'Canis lupus familiaris',
    'dolphin':           'Delphinidae',
    'elephant':          'Elephantidae',
    'flamingo':          'Phoenicopteridae',
    'giant_panda':       'Ailuropoda melanoleuca',
    'giraffe':           'Giraffa',
    'goat':              'Capra hircus',
    'gorilla':           'Gorilla',
    'great_horned_owl':  'Bubo virginianus',
    'grizzly_bear':      'Ursus arctos horribilis',
    'hippo':             'Hippopotamus amphibius',
    'horse':             'Equus caballus',
    'hummingbird':       'Trochilidae',
    'kangaroo':          'Macropus',
    'koala':             'Phascolarctos cinereus',
    'komodo_dragon':     'Varanus komodoensis',
    'leopard':           'Panthera pardus',
    'lion':              'Panthera leo',
    'mallard_duck':      'Anas platyrhynchos',
    'orangutan':         'Pongo',
    'parrot':            'Psittacidae',
    'peacock':           'Pavo cristatus',
    'pelican':           'Pelecanus',
    'penguin':           'Spheniscidae',
    'pig':               'Sus scrofa domesticus',
    'pigeon':            'Columba livia',
    'polar_bear':        'Ursus maritimus',
    'rabbit':            'Oryctolagus cuniculus',
    'raccoon':           'Procyon lotor',
    'red_fox':           'Vulpes vulpes',
    'rhino':             'Rhinocerotidae',
    'robin':             'Turdus migratorius',
    'sheep':             'Ovis aries',
    'squirrel':          'Sciuridae',
    'tiger':             'Panthera tigris',
    'toucan':            'Ramphastidae',
    'turtle':            'Cheloniidae',
    'white_tailed_deer': 'Odocoileus virginianus',
    'wolf':              'Canis lupus',
    'zebra':             'Equus quagga',
}

LABELS = sorted(INAT_NAMES.keys())
print(f'{len(LABELS)} species:')
for i, l in enumerate(LABELS):
    print(f'  {i:2d}: {l}')

In [ ]:
# ── 3. Download images from iNaturalist ──────────────────────────────────────
import os, requests, time
from pathlib import Path

IMAGES_PER_CLASS = 200
TRAIN_DIR = 'dataset/train'

def download_from_inaturalist(taxon_name, save_dir, max_num=200):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    existing = len(os.listdir(save_dir))
    if existing >= max_num * 0.8:
        print(f'  skip {taxon_name} (already has {existing})')
        return existing

    downloaded = 0
    page = 1
    per_page = 50

    while downloaded < max_num:
        try:
            resp = requests.get(
                'https://api.inaturalist.org/v1/observations',
                params={
                    'taxon_name': taxon_name,
                    'per_page': per_page,
                    'page': page,
                    'photos': 'true',
                    'quality_grade': 'research',
                    'order_by': 'votes',
                },
                headers={'User-Agent': 'WildDex/1.0'},
                timeout=10
            )
            results = resp.json().get('results', [])
            if not results:
                break

            for obs in results:
                if downloaded >= max_num:
                    break
                try:
                    photo_url = obs['photos'][0]['url'].replace('square', 'medium')
                    img = requests.get(photo_url, timeout=8).content
                    if len(img) < 5000:
                        continue
                    with open(f'{save_dir}/{downloaded:04d}.jpg', 'wb') as f:
                        f.write(img)
                    downloaded += 1
                except:
                    pass

            page += 1
            time.sleep(0.5)

        except Exception as e:
            print(f'    error on page {page}: {e}')
            break

    print(f'  {taxon_name}: {downloaded} images')
    return downloaded

for label, taxon in INAT_NAMES.items():
    download_from_inaturalist(taxon, f'{TRAIN_DIR}/{label}', IMAGES_PER_CLASS)

print('\nDownload complete.')

In [ ]:
# ── 4. Verify downloads ──────────────────────────────────────────────────────
print(f'{"Species":<25} {"Count":>6}')
print('-' * 33)
total = 0
low = []
for label in LABELS:
    d = f'{TRAIN_DIR}/{label}'
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    total += n
    flag = ' ⚠️ low' if n < 100 else ''
    print(f'{label:<25} {n:>6}{flag}')
    if n < 100:
        low.append(label)
print(f'\nTotal: {total} images')
if low:
    print(f'Low image classes: {low}')

In [ ]:
# ── 5. Build train/val datasets ──────────────────────────────────────────────
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(LABELS)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

class_names = train_ds.class_names
print('Class order matches LABELS:', class_names == LABELS)
print('Classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# ── 6. Build EfficientNetB0 model ────────────────────────────────────────────
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = data_augmentation(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# ── 7. Phase 1: Train head only (base frozen) ────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
]

print('=== Phase 1: Training head (base frozen) ===')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
# ── 8. Phase 2: Fine-tune top layers ─────────────────────────────────────────
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks2 = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
]

print('=== Phase 2: Fine-tuning top layers ===')
h2 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks2)

model.save('wilddex_model.keras')
print('Model saved.')

In [ ]:
# ── 9. Plot training results ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

acc  = h1.history['accuracy']     + h2.history['accuracy']
vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss = h1.history['loss']         + h2.history['loss']
vloss= h1.history['val_loss']     + h2.history['val_loss']
split = len(h1.history['accuracy'])
epochs = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(epochs, acc,  label='Train')
plt.plot(epochs, vacc, label='Val')
plt.axvline(split, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, loss,  label='Train')
plt.plot(epochs, vloss, label='Val')
plt.axvline(split, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Loss'); plt.legend()
plt.tight_layout()
plt.savefig('training_results.png')
plt.show()

val_loss, val_acc = model.evaluate(val_ds)
print(f'\nFinal val accuracy: {val_acc*100:.1f}%')

In [ ]:
# ── 10. Convert to TFLite (float16) ──────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('wilddex_model.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'TFLite model saved — {size_mb:.1f} MB')

In [ ]:
# ── 11. Save labels.txt ───────────────────────────────────────────────────────
with open('labels.txt', 'w') as f:
    f.write('\n'.join(LABELS))

print('labels.txt:')
print('\n'.join(f'{i}: {l}' for i, l in enumerate(LABELS)))

In [ ]:
# ── 12. Download files ────────────────────────────────────────────────────────
from google.colab import files
files.download('wilddex_model.tflite')
files.download('labels.txt')
files.download('training_results.png')
print('Done! Add wilddex_model.tflite and labels.txt to your app.')